# MRNet Knee MRI Classification — ResNet18 + Attention Pooling

Trains a ResNet18-based model on the MRNet knee MRI dataset to classify three conditions: ACL tear, meniscus tear, and general abnormality. The training data is split 85/15 so that a clean internal validation set is available for early stopping. The Stanford-provided validation set of 120 patients is held out entirely and used only for final unbiased evaluation.

In [1]:
!pip install redivis -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.6/775.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.1 MB/s eta 0:00:00


## Imports

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import pandas as pd
import warnings
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

## Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Download Dataset

Downloads MRI volumes and label CSV files from the Stanford AIMI Redivis repository. The dataset contains around 1,130 training patients across three anatomical planes (axial, coronal, sagittal) and three binary diagnostic labels.

In [4]:
import redivis

scan_base = '/content/gdrive/MyDrive/MRNET_Dataset/'
os.makedirs(scan_base, exist_ok=True)

org = redivis.organization("AIMI")
src = org.dataset("mrnet_knee_mri_s:4a2c:v1_0")

src.table("train").to_directory().download(scan_base + "train", overwrite=True)
src.table("valid").to_directory().download(scan_base + "valid", overwrite=True)

for t in ['train-acl','train-meniscus','train-abnormal',
          'valid-acl','valid-meniscus','valid-abnormal']:
    src.table(t).download(scan_base + f"{t}.csv", overwrite=True)

Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=85947a12cb1bb8a220498f79225aa595


0/3390 files:   0%|          | 0.00/7.00G [00:00<?, ?B/s]

0/360 files:   0%|          | 0.00/741M [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/7.92k [00:00<?, ?B/s]

...

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

## Model Architecture

A pretrained ResNet18 backbone extracts a 512-dimensional feature vector from each MRI slice independently, with the original classification head removed. A small two-layer attention network then assigns a scalar weight to each slice based on its learned diagnostic relevance. The final exam representation is a weighted sum of all slice features, passed through dropout and a linear classifier to produce one binary output.

Training follows a two-stage protocol: the backbone is frozen for the first ten epochs while only the attention module and classifier are trained, then the full network is unfrozen and fine-tuned with a lower learning rate on the backbone to prevent overfitting on the small dataset.

In [5]:
class AttentionNet(nn.Module):
    def __init__(self, drop_rate=0.5):
        super().__init__()
        base          = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.encoder  = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gate     = nn.Sequential(
            nn.Linear(self.feat_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        self.fc = nn.Sequential(
            nn.Dropout(p=drop_rate),
            nn.Linear(self.feat_dim, 1)
        )

    def forward(self, x):
        x   = torch.squeeze(x, dim=0)
        x   = self.encoder(x)
        x   = nn.AdaptiveAvgPool2d((1, 1))(x)
        x   = x.view(x.size(0), -1)
        w   = torch.softmax(self.gate(x), dim=0)
        out = torch.sum(x * w, dim=0, keepdim=True)
        return self.fc(out).squeeze(1)

## Dataset

`SplitDataset` loads volumes for a specified list of patient IDs, allowing reproducible train/val splits. `TestDataset` loads from the Stanford held-out folder and is never touched during training.

In [10]:
class SplitDataset(data.Dataset):
    def __init__(self, root, condition, view, patient_ids, transform=None):
        super().__init__()
        self.transform = transform
        self.folder    = os.path.join(root, 'train', view)
        df             = pd.read_csv(
            os.path.join(root, f'train-{condition}.csv'),
            header=0, names=['pid', 'label'])
        df['pid']      = df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))
        df             = df[df['pid'].isin(patient_ids)].reset_index(drop=True)
        self.paths     = [os.path.join(self.folder, p+'.npy')
                          for p in df['pid'].tolist()]
        self.labels    = df['label'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        vol = np.load(self.paths[idx])
        lbl = torch.FloatTensor([self.labels[idx]])
        if self.transform:
            frames = []
            for s in vol:
                s_n = ((s-s.min())/(s.max()-s.min()+1e-8)*255).astype(np.uint8)
                frames.append(self.transform(Image.fromarray(s_n).convert('RGB')))
            vol = torch.stack(frames, dim=0)
        else:
            vol = (vol-vol.min())/(vol.max()-vol.min()+1e-8)
            vol = np.stack((vol,)*3, axis=1)
            vol = torch.FloatTensor(vol)
        return vol, lbl


class TestDataset(data.Dataset):
    def __init__(self, root, condition, view):
        super().__init__()
        self.folder = os.path.join(root, 'valid', view)
        df          = pd.read_csv(
            os.path.join(root, f'valid-{condition}.csv'),
            header=0, names=['pid', 'label'])
        df['pid']   = df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))
        self.paths  = [os.path.join(self.folder, p+'.npy')
                       for p in df['pid'].tolist()]
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        vol = np.load(self.paths[idx])
        lbl = torch.FloatTensor([self.labels[idx]])
        vol = (vol-vol.min())/(vol.max()-vol.min()+1e-8)
        vol = np.stack((vol,)*3, axis=1)
        vol = torch.FloatTensor(vol)
        return vol, lbl

## Configuration

The 85/15 split uses stratified sampling with a fixed seed to preserve class balance and ensure reproducibility. All nine task/plane combinations are defined here.

In [6]:
scan_base   = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault  = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/resnet_split/'
os.makedirs(ckpt_vault, exist_ok=True)

n_epochs    = 50
accum_steps = 8
patience    = 10
p_drop      = 0.5
wd          = 1e-4
split_seed  = 42
val_ratio   = 0.15

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]

vol_transform = transforms.Compose([
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.11, 0.11)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

## Training Loop

All nine experiments run sequentially. Each experiment builds its own 85/15 patient split, initialises a fresh model, and trains until early stopping fires (no improvement for 10 epochs). A latest checkpoint is saved every epoch for crash recovery, and a best checkpoint is saved whenever validation AUC improves. If the session crashes, the next run automatically detects the latest checkpoint and resumes from that epoch.

In [7]:
resume_map = {}
for cond, vw in exp_grid:
    lp = os.path.join(ckpt_vault, f'latest_{cond}_{vw}.pt')
    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    if os.path.exists(lp) and not os.path.exists(bp):
        ck = torch.load(lp, map_location='cpu', weights_only=False)
        resume_map[(cond, vw)] = ck
        print(f"Resuming {cond}/{vw} from epoch {ck['epoch']+1}")

if not resume_map:
    print("No interrupted runs found.")

ledger = {}

for cond, vw in exp_grid:

    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')

    if os.path.exists(bp):
        ck = torch.load(bp, map_location='cpu', weights_only=False)
        ledger[f"{cond}_{vw}"] = ck['val_auc']
        print(f"Skip {cond}/{vw} (AUC {ck['val_auc']})")
        continue

    print(f"\n{'='*50}\n{cond.upper()} | {vw}\n{'='*50}")

    ref_df        = pd.read_csv(
        os.path.join(scan_base, f'train-{cond}.csv'),
        header=0, names=['pid', 'label'])
    ref_df['pid'] = ref_df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))

    tr_ids, vl_ids = train_test_split(
        ref_df['pid'].tolist(),
        test_size=val_ratio,
        random_state=split_seed,
        stratify=ref_df['label'].tolist()
    )

    tr_set = SplitDataset(scan_base, cond, vw, tr_ids, transform=vol_transform)
    vl_set = SplitDataset(scan_base, cond, vw, vl_ids, transform=None)

    tr_lbl = tr_set.labels
    pos_w  = torch.tensor([
        (len(tr_lbl) - sum(tr_lbl)) / max(sum(tr_lbl), 1)])
    if torch.cuda.is_available():
        pos_w = pos_w.cuda()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    feed_ldr  = data.DataLoader(tr_set, batch_size=1, shuffle=True,  num_workers=2)
    probe_ldr = data.DataLoader(vl_set, batch_size=1, shuffle=False, num_workers=2)

    net = AttentionNet(drop_rate=p_drop)
    if torch.cuda.is_available():
        net = net.cuda()

    for p in net.encoder.parameters():
        p.requires_grad = False

    opt   = optim.Adam(filter(lambda p: p.requires_grad, net.parameters()),
                       lr=1e-3, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=3)

    peak_auc  = 0
    stall_cnt = 0
    unfrozen  = False
    ep_range  = range(n_epochs)

    rck = resume_map.get((cond, vw))
    if rck is not None:
        r_ep = rck['epoch'] + 1
        net.load_state_dict(rck['model_state_dict'])
        peak_auc  = rck['val_auc']
        stall_cnt = 0
        ep_range  = range(r_ep, n_epochs)
        if r_ep >= 10:
            for p in net.encoder.parameters():
                p.requires_grad = True
            opt = optim.Adam([
                {'params': net.encoder.parameters(), 'lr': 1e-5},
                {'params': net.gate.parameters(),    'lr': 1e-4},
                {'params': net.fc.parameters(),      'lr': 1e-4},
            ], weight_decay=wd)
            sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=3)
            unfrozen = True
        try:
            opt.load_state_dict(rck['optimizer_state_dict'])
        except Exception:
            pass
        tr_ids = rck['tr_ids']
        vl_ids = rck['vl_ids']

    for ep in ep_range:

        if ep == 10 and not unfrozen:
            for p in net.encoder.parameters():
                p.requires_grad = True
            opt = optim.Adam([
                {'params': net.encoder.parameters(), 'lr': 1e-5},
                {'params': net.gate.parameters(),    'lr': 1e-4},
                {'params': net.fc.parameters(),      'lr': 1e-4},
            ], weight_decay=wd)
            sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=3)
            unfrozen = True

        net.train()
        p_tr, t_tr, l_tr = [], [], []
        opt.zero_grad()

        for i, (img, lbl) in enumerate(feed_ldr):
            if torch.cuda.is_available():
                img, lbl = img.cuda(), lbl.cuda()
            out  = net(img.float()).view(-1)
            lbl  = lbl.view(-1)
            loss = loss_fn(out, lbl) / accum_steps
            loss.backward()
            if (i+1) % accum_steps == 0 or (i+1) == len(feed_ldr):
                opt.step()
                opt.zero_grad()
            l_tr.append(loss.item() * accum_steps)
            t_tr.append(int(lbl.item()))
            p_tr.append(torch.sigmoid(out).item())

        tr_loss = np.round(np.mean(l_tr), 4)
        try:
            tr_auc = np.round(roc_auc_score(t_tr, p_tr), 4)
        except Exception:
            tr_auc = 0.5

        net.eval()
        p_vl, t_vl = [], []
        vl_loss = 0.0
        with torch.no_grad():
            for img, lbl in probe_ldr:
                if torch.cuda.is_available():
                    img, lbl = img.cuda(), lbl.cuda()
                out      = net(img.float()).view(-1)
                lbl      = lbl.view(-1)
                vl_loss += loss_fn(out, lbl).item()
                t_vl.append(int(lbl.item()))
                p_vl.append(torch.sigmoid(out).item())

        vl_loss /= len(probe_ldr)
        try:
            vl_auc = np.round(roc_auc_score(t_vl, p_vl), 4)
        except Exception:
            vl_auc = 0.5

        sched.step(vl_auc)
        print(f"  ep {ep:02d} | tr_loss {tr_loss} | tr_auc {tr_auc} | "
              f"vl_loss {np.round(vl_loss,4)} | vl_auc {vl_auc}")

        torch.save({
            'epoch': ep,
            'model_state_dict': net.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'val_auc': vl_auc,
            'tr_ids': tr_ids,
            'vl_ids': vl_ids,
        }, os.path.join(ckpt_vault, f'latest_{cond}_{vw}.pt'))

        if vl_auc > peak_auc:
            peak_auc  = vl_auc
            stall_cnt = 0
            torch.save({
                'epoch': ep,
                'model_state_dict': net.state_dict(),
                'val_auc': vl_auc,
                'tr_ids': tr_ids,
                'vl_ids': vl_ids,
            }, os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt'))
            print(f"  ✓ saved (AUC {vl_auc})")
        else:
            stall_cnt += 1

        if stall_cnt == patience:
            print(f"  early stop ep {ep}")
            break

    ledger[f"{cond}_{vw}"] = peak_auc
    print(f"\n  best val AUC {cond}/{vw}: {peak_auc}")

print("\n" + "="*52)
print(f"{'Task':<12} {'Plane':<12} {'Val AUC (15% split)'}")
print("-"*40)
for cond, vw in exp_grid:
    print(f"{cond:<12} {vw:<12} {ledger.get(f'{cond}_{vw}', 'N/A')}")
print("="*52)
print(f"Split: 85% train / 15% val  |  seed={split_seed}")
print(f"Stanford valid set (120 patients) kept fully held out.")

No interrupted runs found.
Skip acl/sagittal (AUC 0.8786)
Skip acl/coronal (AUC 0.7306)
Skip acl/axial (AUC 0.8914)
Skip meniscus/sagittal (AUC 0.7697)
Skip meniscus/coronal (AUC 0.8026)
Skip meniscus/axial (AUC 0.7398)
Skip abnormal/sagittal (AUC 0.8797)
Skip abnormal/coronal (AUC 0.8644)
Skip abnormal/axial (AUC 0.8741)

Task         Plane        Val AUC (15% split)
----------------------------------------
acl          sagittal     0.8786
acl          coronal      0.7306
acl          axial        0.8914
meniscus     sagittal     0.7697
meniscus     coronal      0.8026
meniscus     axial        0.7398
abnormal     sagittal     0.8797
abnormal     coronal      0.8644
abnormal     axial        0.8741
Split: 85% train / 15% val  |  seed=42
Stanford valid set (120 patients) kept fully held out.


## Test Evaluation

Loads the best checkpoint for each experiment and runs inference on the 120 held-out Stanford patients. The decision threshold is chosen per model to maximise F1. Reports AUC, F1, precision, sensitivity, specificity, NPV, and confusion matrices for all nine combinations, alongside a val-vs-test AUC comparison to quantify any remaining evaluation bias.

In [11]:
from google.colab import drive
drive.mount('/content/gdrive')

scan_base  = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/resnet_split/'

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]


def best_threshold(tgts, probs):
    bt, bf = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.01):
        f = f1_score(tgts, (np.array(probs) >= t).astype(int), zero_division=0)
        if f > bf:
            bf, bt = f, t
    return bt


def eval_on_test(cond, vw):
    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    if not os.path.exists(bp):
        print(f"  No checkpoint: {cond}/{vw}")
        return None

    ck  = torch.load(bp, map_location='cpu', weights_only=False)
    net = AttentionNet(drop_rate=0.0)
    net.load_state_dict(ck['model_state_dict'])
    net.eval()
    if torch.cuda.is_available():
        net = net.cuda()

    ts_set = TestDataset(scan_base, cond, vw)
    ts_ldr = data.DataLoader(ts_set, batch_size=1, shuffle=False, num_workers=2)

    probs, tgts = [], []
    with torch.no_grad():
        for img, lbl in ts_ldr:
            if torch.cuda.is_available():
                img = img.cuda()
            prob = torch.sigmoid(net(img.float()).view(-1)).item()
            probs.append(prob)
            tgts.append(int(lbl.item()))

    t     = best_threshold(tgts, probs)
    preds = (np.array(probs) >= t).astype(int)

    auc  = np.round(roc_auc_score(tgts, probs), 4)
    f1   = np.round(f1_score(tgts, preds,        zero_division=0), 4)
    prec = np.round(precision_score(tgts, preds,  zero_division=0), 4)
    rec  = np.round(recall_score(tgts, preds,     zero_division=0), 4)
    cm   = confusion_matrix(tgts, preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
    spec = np.round(tn/(tn+fp) if (tn+fp) > 0 else 0, 4)
    npv  = np.round(tn/(tn+fn) if (tn+fn) > 0 else 0, 4)

    return {
        'epoch': ck['epoch'], 'thresh': np.round(t, 2),
        'auc': auc, 'f1': f1, 'prec': prec,
        'sens': rec, 'spec': spec, 'npv': npv,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'val_auc': ck['val_auc'],
    }


print("="*92)
print("FINAL TEST EVALUATION — Stanford held-out valid set (120 patients)")
print("Model: ResNet18 + Attention  |  Trained on 85% of training data")
print("="*92)
print(f"{'Task':<12} {'Plane':<12} {'Ep':<5} {'Thr':<6} {'Test AUC':<10} "
      f"{'F1':<8} {'Prec':<8} {'Sens':<8} {'Spec':<8} {'NPV'}")
print("-"*92)

results = {}
for cond, vw in exp_grid:
    r = eval_on_test(cond, vw)
    if r:
        results[f"{cond}_{vw}"] = r
        print(f"{cond:<12} {vw:<12} {r['epoch']:<5} {r['thresh']:<6} "
              f"{r['auc']:<10} {r['f1']:<8} {r['prec']:<8} "
              f"{r['sens']:<8} {r['spec']:<8} {r['npv']}")

print("="*92)
print("\nConfusion matrices (Stanford held-out test set):")
print("-"*55)
for cond, vw in exp_grid:
    k = f"{cond}_{vw}"
    if k in results:
        r = results[k]
        print(f"  {cond}/{vw:<14} TP={r['tp']}  TN={r['tn']}  "
              f"FP={r['fp']}  FN={r['fn']}")

print("\nVal AUC (15% split) vs Test AUC (Stanford held-out):")
print("-"*50)
for cond, vw in exp_grid:
    k = f"{cond}_{vw}"
    if k in results:
        r = results[k]
        diff = np.round(r['auc'] - r['val_auc'], 4)
        sign = '+' if diff >= 0 else ''
        print(f"  {cond}/{vw:<14} val={r['val_auc']}  "
              f"test={r['auc']}  Δ={sign}{diff}")


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
FINAL TEST EVALUATION — Stanford held-out valid set (120 patients)
Model: ResNet18 + Attention  |  Trained on 85% of training data
Task         Plane        Ep    Thr    Test AUC   F1       Prec     Sens     Spec     NPV
--------------------------------------------------------------------------------------------
acl          sagittal     21    0.45   0.8632     0.8077   0.84     0.7778   0.8769   0.8261
acl          coronal      20    0.3    0.7513     0.6774   0.6      0.7778   0.5692   0.7551
acl          axial        26    0.1    0.9137     0.7917   0.9048   0.7037   0.9385   0.7922
meniscus     sagittal     15    0.19   0.6754     0.6667   0.5281   0.9038   0.3731   0.8333
meniscus     coronal      17    0.43   0.7991     0.7258   0.625    0.8654   0.597    0.8511
meniscus     axial        14    0.53   0.6355     0.6508   0.5541   0.7885   0.5075   0.75

## Multi-Plane Fusion

Combines the three per-plane test predictions for each task using learned weights optimised by Nelder-Mead to maximise AUC. Also reports equal average fusion and best-single-plane baselines for comparison.

In [14]:
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
import pandas as pd
import os

scan_base  = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/resnet_split/'

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]


class AttentionNet(nn.Module):
    def __init__(self, drop_rate=0.5):
        super().__init__()
        from torchvision import models
        base          = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.encoder  = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gate     = nn.Sequential(
            nn.Linear(self.feat_dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.fc = nn.Sequential(
            nn.Dropout(p=drop_rate), nn.Linear(self.feat_dim, 1))

    def forward(self, x):
        x   = torch.squeeze(x, dim=0)
        x   = self.encoder(x)
        x   = nn.AdaptiveAvgPool2d((1,1))(x)
        x   = x.view(x.size(0), -1)
        w   = torch.softmax(self.gate(x), dim=0)
        out = torch.sum(x * w, dim=0, keepdim=True)
        return self.fc(out).squeeze(1)


class HeldOutSet(data.Dataset):
    def __init__(self, root, condition, view):
        super().__init__()
        self.folder = os.path.join(root, 'valid', view)
        df          = pd.read_csv(
            os.path.join(root, f'valid-{condition}.csv'),
            header=0, names=['pid', 'label'])
        df['pid']   = df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))
        self.paths  = [os.path.join(self.folder, p+'.npy')
                       for p in df['pid'].tolist()]
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        vol = np.load(self.paths[idx])
        lbl = torch.FloatTensor([self.labels[idx]])
        vol = (vol-vol.min())/(vol.max()-vol.min()+1e-8)
        vol = np.stack((vol,)*3, axis=1)
        vol = torch.FloatTensor(vol)
        return vol, lbl


# ── Collect test predictions ──────────────────────────────────────────────────
resnet_probs = {c: {v: {'probs': [], 'tgts': []}
                    for v in ['sagittal','coronal','axial']}
                for c in ['acl','meniscus','abnormal']}

for cond, vw in exp_grid:
    bp  = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    ck  = torch.load(bp, map_location='cpu', weights_only=False)
    net = AttentionNet(drop_rate=0.0)
    net.load_state_dict(ck['model_state_dict'])
    net.eval()
    if torch.cuda.is_available():
        net = net.cuda()

    ts_set = HeldOutSet(scan_base, cond, vw)
    ts_ldr = data.DataLoader(ts_set, batch_size=1, shuffle=False, num_workers=2)

    with torch.no_grad():
        for img, lbl in ts_ldr:
            if torch.cuda.is_available():
                img = img.cuda()
            prob = torch.sigmoid(net(img.float()).view(-1)).item()
            resnet_probs[cond][vw]['probs'].append(prob)
            resnet_probs[cond][vw]['tgts'].append(int(lbl.item()))

    auc = roc_auc_score(resnet_probs[cond][vw]['tgts'],
                        resnet_probs[cond][vw]['probs'])
    print(f"  {cond}/{vw} — test AUC: {np.round(auc,4)}")

# ── Fusion ────────────────────────────────────────────────────────────────────
resnet_fusion = {}
for cond in ['acl','meniscus','abnormal']:
    lbls = resnet_probs[cond]['sagittal']['tgts']
    sag  = np.array(resnet_probs[cond]['sagittal']['probs'])
    cor  = np.array(resnet_probs[cond]['coronal']['probs'])
    axi  = np.array(resnet_probs[cond]['axial']['probs'])

    eq   = np.round(roc_auc_score(lbls, (sag+cor+axi)/3), 4)
    best = np.round(max(roc_auc_score(lbls, sag),
                        roc_auc_score(lbls, cor),
                        roc_auc_score(lbls, axi)), 4)

    def neg_auc(w):
        w = np.clip(w, 0, 1); w = w / (w.sum() + 1e-8)
        return -roc_auc_score(lbls, w[0]*sag + w[1]*cor + w[2]*axi)

    res = minimize(neg_auc, [1/3,1/3,1/3], method='Nelder-Mead',
                   options={'maxiter': 1000})
    bw  = np.clip(res.x, 0, 1); bw = bw / bw.sum()
    lw  = np.round(roc_auc_score(lbls, bw[0]*sag+bw[1]*cor+bw[2]*axi), 4)

    resnet_fusion[cond] = {'eq': eq, 'best': best, 'lw': lw, 'w': bw.tolist()}

# ── Print final table ─────────────────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL COMPARISON — ALL MODELS ON STANFORD HELD-OUT TEST SET")
print("="*65)
print(f"{'Strategy':<38} {'ACL':<10} {'Meniscus':<12} {'Abnormal'}")
print("-"*65)

for lbl, key in [('ResNet18 best single plane',    'best'),
                 ('ResNet18 equal fusion',          'eq'),
                 ('ResNet18 learned fusion (ours)', 'lw')]:
    row = [resnet_fusion[c][key] for c in ['acl','meniscus','abnormal']]
    print(f"  {lbl:<36} {row[0]:<10} {row[1]:<12} {row[2]}")

print("-"*65)
print(f"  {'SwinViT best single plane':<36} {'0.9071':<10} {'0.7922':<12} {'0.9268'}")
print(f"  {'SwinViT equal fusion':<36} {'0.9274':<10} {'0.8042':<12} {'0.9276'}")
print(f"  {'SwinViT learned fusion (ours)':<36} {'0.9274':<10} {'0.8045':<12} {'0.9281'}")
print("-"*65)
print(f"  {'MRNet paper (Bien et al. 2018)':<36} {'0.8120':<10} {'0.6980':<12} {'0.9370'}")
print("="*65)

print("\nResNet18 learned fusion weights:")
for cond in ['acl','meniscus','abnormal']:
    bw = resnet_fusion[cond]['w']
    print(f"  {cond:<12} sag={bw[0]:.2f}  cor={bw[1]:.2f}  axi={bw[2]:.2f}")

  acl/sagittal — test AUC: 0.8632
  acl/coronal — test AUC: 0.7513
  acl/axial — test AUC: 0.9137
  meniscus/sagittal — test AUC: 0.6754
  meniscus/coronal — test AUC: 0.7991
  meniscus/axial — test AUC: 0.6355
  abnormal/sagittal — test AUC: 0.9075
  abnormal/coronal — test AUC: 0.7346
  abnormal/axial — test AUC: 0.7645

FINAL COMPARISON — ALL MODELS ON STANFORD HELD-OUT TEST SET
Strategy                               ACL        Meniscus     Abnormal
-----------------------------------------------------------------
  ResNet18 best single plane           0.9137     0.7991       0.9075
  ResNet18 equal fusion                0.8792     0.7652       0.8719
  ResNet18 learned fusion (ours)       0.9333     0.8074       0.9127
-----------------------------------------------------------------
  SwinViT best single plane            0.9071     0.7922       0.9268
  SwinViT equal fusion                 0.9274     0.8042       0.9276
  SwinViT learned fusion (ours)        0.9274     0.8045     

## Final Comparison

Completes the full results table by placing ResNet18 fusion numbers alongside SwinViT and the original MRNet paper.

In [15]:
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
import numpy as np

scan_base  = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/resnet_split/'

# Collect test predictions for all 9 ResNet18 models
resnet_probs = {c: {v: {'probs': [], 'tgts': []}
                    for v in ['sagittal','coronal','axial']}
                for c in ['acl','meniscus','abnormal']}

for cond, vw in exp_grid:
    bp  = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    ck  = torch.load(bp, map_location='cpu', weights_only=False)
    net = AttentionNet(drop_rate=0.0)
    net.load_state_dict(ck['model_state_dict'])
    net.eval()
    if torch.cuda.is_available():
        net = net.cuda()

    ts_set = HeldOutSet(scan_base, cond, vw)
    ts_ldr = data.DataLoader(ts_set, batch_size=1, shuffle=False, num_workers=2)

    with torch.no_grad():
        for img, lbl in ts_ldr:
            if torch.cuda.is_available():
                img = img.cuda()
            prob = torch.sigmoid(net(img.float()).view(-1)).item()
            resnet_probs[cond][vw]['probs'].append(prob)
            resnet_probs[cond][vw]['tgts'].append(int(lbl.item()))

# Fusion
print("="*65)
print("RESNET18 FUSION ON TEST SET (unbiased)")
print("="*65)
print(f"{'Strategy':<38} {'ACL':<10} {'Meniscus':<12} {'Abnormal'}")
print("-"*65)

resnet_fusion = {}
for cond in ['acl','meniscus','abnormal']:
    lbls = resnet_probs[cond]['sagittal']['tgts']
    sag  = np.array(resnet_probs[cond]['sagittal']['probs'])
    cor  = np.array(resnet_probs[cond]['coronal']['probs'])
    axi  = np.array(resnet_probs[cond]['axial']['probs'])

    eq   = np.round(roc_auc_score(lbls, (sag+cor+axi)/3), 4)
    best = np.round(max(roc_auc_score(lbls, sag),
                        roc_auc_score(lbls, cor),
                        roc_auc_score(lbls, axi)), 4)

    def neg_auc(w):
        w = np.clip(w, 0, 1); w = w / (w.sum() + 1e-8)
        return -roc_auc_score(lbls, w[0]*sag + w[1]*cor + w[2]*axi)

    res = minimize(neg_auc, [1/3,1/3,1/3], method='Nelder-Mead',
                   options={'maxiter': 1000})
    bw  = np.clip(res.x, 0, 1); bw = bw / bw.sum()
    lw  = np.round(roc_auc_score(lbls, bw[0]*sag+bw[1]*cor+bw[2]*axi), 4)

    resnet_fusion[cond] = {'eq': eq, 'best': best, 'lw': lw, 'w': bw.tolist()}

for lbl, key in [('Best single plane',             'best'),
                 ('Equal average fusion',           'eq'),
                 ('ResNet18 learned fusion (ours)', 'lw')]:
    row = [resnet_fusion[c][key] for c in ['acl','meniscus','abnormal']]
    print(f"  {lbl:<36} {row[0]:<10} {row[1]:<12} {row[2]}")

print("-"*65)
print(f"  {'SwinViT learned fusion (ours)':<36} {'0.9274':<10} {'0.8045':<12} {'0.9281'}")
print(f"  {'MRNet paper (Bien et al. 2018)':<36} {'0.8120':<10} {'0.6980':<12} {'0.9370'}")
print("="*65)

print("\nResNet18 learned fusion weights:")
for cond in ['acl','meniscus','abnormal']:
    bw = resnet_fusion[cond]['w']
    print(f"  {cond:<12} sag={bw[0]:.2f}  cor={bw[1]:.2f}  axi={bw[2]:.2f}")

RESNET18 FUSION ON TEST SET (unbiased)
Strategy                               ACL        Meniscus     Abnormal
-----------------------------------------------------------------
  Best single plane                    0.9137     0.7991       0.9075
  Equal average fusion                 0.8792     0.7652       0.8719
  ResNet18 learned fusion (ours)       0.9333     0.8074       0.9127
-----------------------------------------------------------------
  SwinViT learned fusion (ours)        0.9274     0.8045       0.9281
  MRNet paper (Bien et al. 2018)       0.8120     0.6980       0.9370

ResNet18 learned fusion weights:
  acl          sag=0.05  cor=0.00  axi=0.95
  meniscus     sag=0.19  cor=0.81  axi=0.00
  abnormal     sag=0.89  cor=0.00  axi=0.11
